# Phase 2 : exploration des données (P2-01)

Ce notebook décrit les fichiers tels qu'ils sont reçus, sans rien modifier : taille, colonnes, types, exemples et clés entre les tables. Les contrôles de qualité sont dans `02_diagnostic_qualite.ipynb`.

Critère de fin de P2-01 : notebook exécuté et versionné, avec des constats sous chaque résultat.

Avant de commiter : **Clear All Outputs**, car certaines sorties peuvent afficher des données sensibles.

## Chargement des données

Les fichiers sont lus avec les fonctions communes de `src/load_data.py` (P3-01). Les étiquettes de fraude sont lues avec `load_fraud_labels` (P2-17), qui transforme Yes/No en 1/0.

In [ ]:
import pandas as pd

from src.load_data import (
    DATA_DIR, load_transactions, load_cards, load_users, load_mcc_codes, load_fraud_labels,
)
from src.diagnostic import describe_files, describe_columns, find_key_columns

pd.set_option("display.max_colwidth", None)

In [ ]:
df_users = load_users()
df_card = load_cards()
df_transaction = load_transactions()
df_mcc = load_mcc_codes()
df_labels = load_fraud_labels()

tables = {"users": df_users, "cards": df_card, "transactions": df_transaction,
          "mcc": df_mcc, "fraud_labels": df_labels}

## Fichiers reçus

In [ ]:
fichiers = {"users": "users_data.csv", "cards": "cards_data.csv",
            "transactions": "transactions_data.csv", "mcc": "mcc_codes.json",
            "fraud_labels": "fraud_labels.json"}
describe_files(tables, fichiers, DATA_DIR)

**Constats**

- Cinq fichiers sont reçus. Le plus gros est `transactions_data.csv` : 1 258,5 Mo pour 13 305 915 lignes et 12 colonnes.
- `fraud_labels.json` fait 159,1 Mo pour 8 914 963 lignes.
- Les trois autres fichiers font 0,5 Mo ou moins : 2 000 clients, 6 146 cartes et 109 codes MCC.

## Colonnes des clients (`users_data.csv`)

La colonne `address` est masquée.

In [ ]:
describe_columns(df_users, sensitive=["address"])

**Constats**

- Chaque client a un identifiant distinct : 2 000 valeurs pour 2 000 lignes.
- `per_capita_income`, `yearly_income` et `total_debt` sont du texte avec le symbole $ (exemple : $59696). Ils seront convertis en P3-02.
- `address` a 1 999 valeurs distinctes pour 2 000 clients : deux clients ont la même adresse. Cette colonne est sensible.
- `gender` a 2 valeurs. `current_age` et `birth_year` ont 80 valeurs chacune.

## Colonnes des cartes (`cards_data.csv`)

Les colonnes `card_number` et `cvv` sont masquées.

In [ ]:
describe_columns(df_card, sensitive=["card_number", "cvv"])

**Constats**

- 6 146 cartes appartiennent à 2 000 clients : chaque client a au moins une carte.
- `expires` et `acct_open_date` sont du texte au format mois/année (exemples : 12/2022 et 09/2002). `credit_limit` est du texte avec le symbole $ (exemple : $24295).
- `card_on_dark_web` n'a qu'une seule valeur : No.
- `card_number` (6 146 valeurs, une par carte) et `cvv` sont sensibles.
- `card_brand` a 4 valeurs, `card_type` 3 et `has_chip` 2.

## Colonnes des transactions (`transactions_data.csv`)

In [ ]:
describe_columns(df_transaction)

**Constats**

- Chaque transaction a un identifiant distinct : 13 305 915 valeurs.
- Les transactions concernent 1 219 clients et 4 071 cartes.
- `date` est déjà lue comme une date. `amount` est du texte avec le symbole $, et l'exemple ($-77.00) montre des montants négatifs.
- Les transactions touchent 74 831 commerçants, 12 492 villes et 199 valeurs de `merchant_state`.
- `use_chip` a 3 canaux. `errors` a 22 valeurs distinctes (exemple : Technical Glitch), détaillées en P2-13.
- `zip` est lu comme un nombre décimal (exemple : 58523.0).
- `mcc` a 109 valeurs, autant que de codes dans `mcc_codes.json`.

## Codes MCC (`mcc_codes.json`) et étiquettes de fraude (`fraud_labels.json`)

In [ ]:
describe_columns(df_mcc)

In [ ]:
describe_columns(df_labels)

**Constats**

- `mcc_codes.json` contient 109 codes mais 108 descriptions distinctes : deux codes ont la même description.
- `fraud_labels.json` contient 8 914 963 identifiants distincts. `is_fraud` a 2 valeurs.

## Clés entre les tables

In [ ]:
find_key_columns(tables)

**Constats**

- `cards` est reliée à `users` par `client_id`.
- `transactions` est reliée à `users` par `client_id`, à `cards` par `card_id` et aux codes MCC par `mcc`.
- `fraud_labels` est reliée à `transactions` par `id`.
- `merchant_id` n'a pas de table de commerçants : il sert seulement à reconnaître un même commerçant.
- Ces liens sont contrôlés en P2-02.

## Suite

Les points repérés ici sont mesurés dans `02_diagnostic_qualite.ipynb`, puis traités dans `03_nettoyage_transformation.ipynb`.

Le sens de chaque colonne est décrit dans `docs/P1-04_dictionnaire_donnees.md`.